In [3]:
%load_ext autoreload
%autoreload 2
import parse_agent,pde_descriptions,system_prompt,coding_agent
import os
from dotenv import load_dotenv
from google import genai
from google.genai.types import GenerateContentConfig
import markdown_to_json
import json
from ollama import chat

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")


fk_description = pde_descriptions.fk_description
tau_d = 0.5714
pde_description = fk_description.format(tau_d=tau_d)

parse_prompt = parse_agent.parse_prompt
system_prompt = system_prompt.system_prompt

user_prompt = parse_prompt.format(
    user_input=pde_description,
)

response = chat(
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)

#deleat ```json at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```json"):
    response_text = response_text[len("```json"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("parsed_resp.json", "w", encoding="utf-8") as f:
    f.write(response_text)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
{
  "PDEs": "\\frac{\\partial u}{\\partial t} = D \\nabla^2 u - \\frac{I_{fi}(u, v) + I_{so}(u) + I_{si}(u, w)}{C_m} \\n\\frac{\\partial v}{\\partial t} = \\begin{cases} \\frac{1-v}{\\tau_{mv}(u)} & u < V_c \\\\ \\frac{-v}{\\tau_{pv}} & u \\ge V_c \\end{cases} \\n\\frac{\\partial w}{\\partial t} = \\begin{cases} \\frac{1-w}{\\tau_{mw}} & u < V_c \\\\ \\frac{-w}{\\tau_{pw}} & u \\ge V_c \\end{cases} \\n\\nI_{fi}(u, v) = \\frac{-v \\cdot H(u - V_c) \\cdot (u - V_c) \\cdot (1 - u)}{\\tau_d} \\n\\nI_{so}(u) = \\frac{u(1 - H(u - V_c))}{\\tau_0} + \\frac{H(u - V_c)}{\\tau_r} \\n\\nI_{si}(u, w) = \\frac{-w(1 + \\tanh(k(u - V_{csi})))}{2\\tau_{si}} \\n\\nH(x) = \\begin{cases} 0 & x < 0 \\\\ 1 & x \\ge 0 \\end{cases}",
  "number_of_state_variables": 3,
  "texture_size": 512,
  "spatial_step": 0.0390625,
  "domain_size": 20.0,
  "temporal_step": 0.025,
  "time_horizon": 100.0,
  "boundary_conditions": "Neuman

In [4]:
# read json
with open("parsed_resp.json", "r") as f:
    parsed_resp = json.load(f)
print(parsed_resp.keys())

dict_keys(['PDEs', 'number_of_state_variables', 'texture_size', 'spatial_step', 'domain_size', 'temporal_step', 'time_horizon', 'boundary_conditions', 'parameter_values', 'notes'])


In [5]:
coding_prompt = coding_agent.coding_prompt
with open('coding_skeleton.frag', 'r') as f:
    coding_skeleton = f.read()

user_prompt = coding_prompt.format(
    PDEs=parsed_resp["PDEs"],
    coding_skeleton = coding_skeleton
)

In [6]:
response = chat(
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)


precision highp float;
precision highp int;

uniform sampler2D   inTexture ;
uniform float       dt, D, dx, dy;

in vec2 cc, pixPos ;

layout (location = 0) out vec4 ocolor ;

#define u       color.r
#define v       color.g
#define w       color.b

const float tau_pv = 7.99;
const float tau_v1 = 9.8;
const float tau_v2 = 312.5;
const float tau_pw = 870.0;
const float tau_mw = 41.0;
const float tau_d  = 0.5714;
const float tau_0  = 12.5;
const float tau_r  = 33.83;
const float tau_si = 29.0;
const float K      = 10.0;
const float V_csi  = 0.861;
const float V_c = 0.13; 
const float V_v = 0.04;
const float C_m    = 1.0;

vec2 size = vec2(textureSize(inTexture, 0));
vec2 ii = vec2(1.,0.)/size;
vec2 jj = vec2(0.,1.)/size;

void main() {
    vec4 color = texture( inTexture , cc ) ;

    float laplacian = (
        texture(inTexture, cc + ii).r + 
        texture(inTexture, cc - ii).r + 
        texture(inTexture, cc + jj).r + 
        texture(inTexture, cc - jj).r - 
        4.0 * u
    ) /

In [ ]:
#deleat ```glsl at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text) 


In [12]:
import debug_agent
import glob,os

log_files = glob.glob("*.log")
for log_file in log_files:
    with open(log_file, 'r', encoding='utf-8') as f:
        original_log_content = f.read()

debug_prompt = debug_agent.debug_prompt.format(
    shader_codes = response.message.content,
    log_info = original_log_content
)

debug_response = chat(
    model='qwen2.5-coder:3b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': debug_prompt},
    ],
)
print(debug_response.message.content)

Based on the error messages you provided, it seems there are syntax errors in your GLSL code. Specifically, there's an issue with the use of the `.` operator in the shader code, which is causing a compilation error. Let's clean up the shader to ensure all syntax is correct and properly linked.

Here's the corrected version of the shader:

```glsl
precision highp float ;
precision highp int ;

uniform sampler2D   inTexture ;
uniform float       dt, D, dx, dy;

in vec2 cc, pixPos ;

layout (location = 0) out vec4 ocolor ;

#define u       color.r
#define v       color.g
#define w       color.b

const float tau_pv = 7.99;
const float tau_v1 = 9.8;
const float tau_v2 = 312.5;
const float tau_pw = 870.0;
const float tau_mw = 41.0;
const float tau_d  = 0.5714;
const float tau_0  = 12.5;
const float tau_r  = 33.83;
const float tau_si = 29.0;
const float K      = 10.0;
const float V_csi  = 0.861;
const float V_c = 0.13; 
const float V_v = 0.04;
const float C_m    = 1.0;

// Function to calcula

In [13]:
#deleat ```glsl at beginning of response.text if exist
response_text = debug_response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text)